# Budgeted LoRA Rank-Pattern Experiments

This notebook extends the GPT-2 Medium / E2E LoRA reimplementation beyond uniform rank sweeps. It keeps the same trainable-parameter budget as the completed uniform `r=4, alpha=32` baseline and redistributes rank by layer or by Q/V projection.

The four new runs are organized as two separate comparisons:

1. Depth allocation: `late_ramp_balanced` vs `early_ramp_balanced`, both balanced Q/V and same total `r=4` budget.
2. Projection allocation: `value_heavy_uniform` vs `query_heavy_uniform`, same Q+V total per layer and no depth ramp.

`late_value_heavy` is intentionally excluded because it would confound depth allocation with Q/V allocation. The existing uniform `r=4` run remains the baseline control.

## 1. Check GPU

Run this first to confirm Colab assigned the expected CUDA GPU. If you requested an A100 runtime, this cell should print an NVIDIA A100 device.

In [ ]:
!nvidia-smi

import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 2. Clone, Install, Drive, And Setup

Run these cells from a clean Colab runtime. They clone or update the project under `/content/CS4782-final-project`, switch into `lora-gpt2-medium-e2e`, install dependencies, mount Google Drive, and then load the libraries used by the rank-pattern experiment.

In [ ]:
from getpass import getpass
from pathlib import Path
import os
import subprocess

REPO_OWNER = 'justinlxiang'
REPO_NAME = 'CS4782-final-project'
BRANCH = 'main'
PROJECT_DIR = Path('/content') / REPO_NAME
WORK_DIR = PROJECT_DIR / 'lora-gpt2-medium-e2e'

token = getpass('GitHub token, or press Enter for public clone: ')
repo_url = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
if token:
    repo_url = f'https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git'

if PROJECT_DIR.exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, repo_url, str(PROJECT_DIR)], check=True)

os.chdir(WORK_DIR)
print('working directory:', Path.cwd())
!git log --oneline -3

In [ ]:
!pip install -q -r requirements.txt

import copy
import json
import os
import shlex
import shutil
import subprocess
from collections import deque
from pathlib import Path

from google.colab import drive
import matplotlib.pyplot as plt
import pandas as pd
import yaml

# Match the other Colab notebooks: keep full run artifacts backed up in Drive.
drive.mount('/content/drive')
DRIVE_PATTERN_DIR = Path('/content/drive/MyDrive/lora_rank_patterns')
DRIVE_PATTERN_DIR.mkdir(parents=True, exist_ok=True)
print('Drive pattern dir:', DRIVE_PATTERN_DIR)

!python -m pytest tests/test_lora_layers.py tests/test_inject.py tests/test_checkpointing.py

## 3. Data Prep

This matches the existing E2E preprocessing path used by the uniform `r=4` and rank-sweep runs.

In [ ]:
!mkdir -p data/raw/e2e
!curl -L -o data/raw/e2e/train.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/train.txt
!curl -L -o data/raw/e2e/valid.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/valid.txt
!curl -L -o data/raw/e2e/test.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/test.txt
!python scripts/prepare_e2e.py --config configs/e2e_gpt2_medium_lora.yaml
!wc -l data/raw/e2e/*.txt data/processed/e2e_gpt2/*.jsonl

## 4. Pattern Matrix

All runs use the same GPT-2 Medium Q/V LoRA trainable-parameter budget as the uniform `r=4` baseline. Layer indices below are 0-based transformer block indices. For each layer, the rank units are:

```text
rank_units[layer] = r_q[layer] + r_v[layer]
total_rank_units = sum_l rank_units[layer] = 192
trainable params = 192 * (1024 + 1024) = 393,216
```

The factor `(1024 + 1024)` is the LoRA A/B parameter cost for one rank unit on a GPT-2 Medium Q or V projection. The source code derives per-slice alpha from the scalar baseline when `alpha_pattern` is omitted, so nonzero slices keep `alpha / r = 32 / 4 = 8`.

### Baseline

| Run | Layers | `r_q` | `r_v` | Units per layer | Total rank units | Trainable params |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| `uniform_r4` | 0-23 | 4 | 4 | 8 | `24 * 8 = 192` | `192 * 2048 = 393,216` |

### Depth Comparison

| Run | Layers | `r_q` | `r_v` | Units per layer | Block contribution | Total rank units | Trainable params |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |
| `late_ramp_balanced` | 0-7 | 2 | 2 | 4 | `8 * 4 = 32` |  |  |
| `late_ramp_balanced` | 8-15 | 4 | 4 | 8 | `8 * 8 = 64` |  |  |
| `late_ramp_balanced` | 16-23 | 6 | 6 | 12 | `8 * 12 = 96` | `32 + 64 + 96 = 192` | `192 * 2048 = 393,216` |
| `early_ramp_balanced` | 0-7 | 6 | 6 | 12 | `8 * 12 = 96` |  |  |
| `early_ramp_balanced` | 8-15 | 4 | 4 | 8 | `8 * 8 = 64` |  |  |
| `early_ramp_balanced` | 16-23 | 2 | 2 | 4 | `8 * 4 = 32` | `96 + 64 + 32 = 192` | `192 * 2048 = 393,216` |

### Projection Comparison

| Run | Layers | `r_q` | `r_v` | Units per layer | Total rank units | Trainable params |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| `value_heavy_uniform` | 0-23 | 2 | 6 | 8 | `24 * 8 = 192` | `192 * 2048 = 393,216` |
| `query_heavy_uniform` | 0-23 | 6 | 2 | 8 | `24 * 8 = 192` | `192 * 2048 = 393,216` |

No conditional combined/post-experiment run is defined in this notebook. If one is added later, list it separately here and label it conditional so it is not mixed into the planned depth or projection comparisons.

In [ ]:
BASELINE_TRAINABLE_PARAMS = 393_216
BASELINE_RANK = 4
BASELINE_ALPHA = 32
GENERATION_BATCH_SIZE = 16
MAX_TRAIN_STEPS = None  # Set to 2000 for a pilot; None means full 5 epochs.
RUN_TRAINING = True
RUN_GENERATION_AND_EVAL = True
RUN_OFFICIAL_E2E = True


def repeated_pattern(blocks):
    pattern = []
    for count, q_rank, v_rank in blocks:
        pattern.extend({'query': q_rank, 'value': v_rank} for _ in range(count))
    assert len(pattern) == 24
    return pattern


PATTERNS = [
    {
        'comparison': 'depth_allocation',
        'pattern_name': 'late_ramp_balanced',
        'description': 'Balanced Q/V; more rank in later layers.',
        'rank_pattern': repeated_pattern([(8, 2, 2), (8, 4, 4), (8, 6, 6)]),
    },
    {
        'comparison': 'depth_allocation',
        'pattern_name': 'early_ramp_balanced',
        'description': 'Balanced Q/V; more rank in earlier layers.',
        'rank_pattern': repeated_pattern([(8, 6, 6), (8, 4, 4), (8, 2, 2)]),
    },
    {
        'comparison': 'projection_allocation',
        'pattern_name': 'value_heavy_uniform',
        'description': 'No depth ramp; value gets more rank than query in every layer.',
        'rank_pattern': repeated_pattern([(24, 2, 6)]),
    },
    {
        'comparison': 'projection_allocation',
        'pattern_name': 'query_heavy_uniform',
        'description': 'No depth ramp; query gets more rank than value in every layer.',
        'rank_pattern': repeated_pattern([(24, 6, 2)]),
    },
]

for spec in PATTERNS:
    rank_units = sum(layer['query'] + layer['value'] for layer in spec['rank_pattern'])
    params = rank_units * (1024 + 1024)
    print(spec['comparison'], spec['pattern_name'], 'rank_units=', rank_units, 'params=', params)
    assert rank_units == 192
    assert params == BASELINE_TRAINABLE_PARAMS

## 5. Create Pattern Configs

The generated configs keep scalar `lora.rank=4` and `lora.alpha=32` for backward compatibility and baseline scale. The new `lora.rank_pattern` field overrides Q/V ranks per layer; omitted `alpha_pattern` means the code derives `alpha = 8 * rank` for each nonzero slice.

In [ ]:
base_config_path = Path('configs/e2e_gpt2_medium_lora.yaml')
base_config = yaml.safe_load(base_config_path.read_text())
pattern_config_paths = []

for spec in PATTERNS:
    cfg = copy.deepcopy(base_config)
    run_name = spec['pattern_name']
    run_dir = Path('outputs/runs/rank_patterns') / run_name

    cfg['project']['output_dir'] = str(run_dir)
    cfg['lora']['rank'] = BASELINE_RANK
    cfg['lora']['alpha'] = BASELINE_ALPHA
    cfg['lora']['rank_pattern'] = spec['rank_pattern']
    cfg['generation']['decoder'] = 'official_beam'
    cfg['generation']['length_penalty'] = 0.9
    cfg['generation']['batch_size'] = GENERATION_BATCH_SIZE
    cfg['evaluation']['predictions_file'] = str(run_dir / 'generations_test.txt')
    cfg['evaluation']['references_file'] = 'data/processed/e2e_gpt2/references_test.txt'

    out_path = Path('configs/rank_patterns') / f"e2e_gpt2_medium_lora_{run_name}.yaml"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
    pattern_config_paths.append((spec, out_path, run_dir))

for spec, config_path, run_dir in pattern_config_paths:
    print(spec['comparison'], spec['pattern_name'], config_path, '->', run_dir)

In [ ]:
for spec, config_path, run_dir in pattern_config_paths:
    print('\n===', spec['pattern_name'], 'parameter count ===')
    result = subprocess.run(
        ['python', 'scripts/count_params.py', '--config', str(config_path)],
        text=True,
        capture_output=True,
        check=True,
    )
    print(result.stdout)
    assert f'expected_gpt2_lora_parameters={BASELINE_TRAINABLE_PARAMS}' in result.stdout

## 6. Train, Generate, And Evaluate

These cells mirror the rank-sweep notebook. Keep decoding/evaluation identical across all pattern runs and compare back to the existing uniform `r=4` baseline. Each pattern run is copied to Google Drive immediately after training, after generation/evaluation, and after the official E2E scorer so completed work survives Colab runtime resets.

In [ ]:
def persist_pattern_run_to_drive(spec, config_path, run_dir, phase):
    DRIVE_PATTERN_DIR.mkdir(parents=True, exist_ok=True)
    pattern_name = spec['pattern_name']

    if config_path.exists():
        config_dest = DRIVE_PATTERN_DIR / 'configs' / config_path.name
        config_dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(config_path, config_dest)
        print(f"[{phase}] Backed up config for {pattern_name}: {config_dest}")

    if run_dir.exists():
        run_dest = DRIVE_PATTERN_DIR / run_dir.name
        shutil.copytree(run_dir, run_dest, dirs_exist_ok=True)
        print(f"[{phase}] Backed up run for {pattern_name}: {run_dest}")
    else:
        print(f"[{phase}] No run directory yet for {pattern_name}: {run_dir}")


def run_checked(cmd, env=None):
    print('+', shlex.join(cmd), flush=True)
    tail = deque(maxlen=80)
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        tail.append(line)
    returncode = process.wait()
    if returncode != 0:
        print('\nCommand failed:', shlex.join(cmd))
        print('Last output lines:')
        print(''.join(tail))
        raise subprocess.CalledProcessError(returncode, cmd)


if RUN_TRAINING:
    for spec, config_path, run_dir in pattern_config_paths:
        print(f"=== Training {spec['pattern_name']} ===")
        cmd = ['python', 'scripts/train.py', '--config', str(config_path), '--train', '--device', 'cuda']
        if MAX_TRAIN_STEPS is not None:
            cmd += ['--max-train-steps', str(MAX_TRAIN_STEPS)]
        run_checked(cmd)
        persist_pattern_run_to_drive(spec, config_path, run_dir, phase='training')
else:
    print('RUN_TRAINING=False; skipping training.')

In [ ]:
if RUN_GENERATION_AND_EVAL:
    env = {**os.environ, 'TOKENIZERS_PARALLELISM': 'false', 'TRANSFORMERS_VERBOSITY': 'error'}
    for spec, config_path, run_dir in pattern_config_paths:
        print(f"\n=== Generating/evaluating {spec['pattern_name']} ===")
        adapter = run_dir / 'checkpoints' / 'adapter_final.pt'
        if not adapter.exists():
            raise FileNotFoundError(f'Missing adapter: {adapter}')

        run_checked([
            'python', 'scripts/generate.py',
            '--config', str(config_path),
            '--split', 'test',
            '--adapter', str(adapter),
            '--batch-size', str(GENERATION_BATCH_SIZE),
        ], env=env)
        run_checked(['python', 'scripts/evaluate.py', '--config', str(config_path)], env=env)
        run_checked([
            'python', 'scripts/make_figures.py',
            '--config', str(config_path),
            '--run-dir', str(run_dir),
            '--figures-dir', str(run_dir / 'figures'),
        ], env=env)
        persist_pattern_run_to_drive(spec, config_path, run_dir, phase='generation/eval')
else:
    print('RUN_GENERATION_AND_EVAL=False; skipping generation/evaluation.')

In [ ]:
if RUN_OFFICIAL_E2E:
    Path('external').mkdir(exist_ok=True)
    if not Path('external/e2e-metrics/.git').exists():
        run_checked(['git', 'clone', 'https://github.com/tuetschek/e2e-metrics.git', 'external/e2e-metrics'])

    for spec, config_path, run_dir in pattern_config_paths:
        ref_file = run_dir / 'generations_test.e2e_refs.txt'
        pred_file = run_dir / 'generations_test.e2e_preds.txt'
        out_file = run_dir / 'generations_test.official_e2e_metrics.txt'
        print(f"\n=== Official E2E metrics {spec['pattern_name']} ===")
        result = subprocess.run(
            ['python', 'external/e2e-metrics/measure_scores.py', str(ref_file), str(pred_file), '-p'],
            text=True,
            capture_output=True,
        )
        out_file.write_text('STDOUT:\n' + result.stdout + '\n\nSTDERR:\n' + result.stderr, encoding='utf-8')
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
        persist_pattern_run_to_drive(spec, config_path, run_dir, phase='official-e2e')
else:
    print('RUN_OFFICIAL_E2E=False; skipping official scorer.')

## 7. Summary

The summary includes the existing uniform `r=4` baseline plus the four new pattern runs. The comparison column keeps the two experimental questions separate.

In [ ]:
def read_validation_summary(run_dir):
    metrics_path = run_dir / 'metrics.jsonl'
    validation = []
    if metrics_path.exists():
        for line in metrics_path.read_text().splitlines():
            if line.strip():
                record = json.loads(line)
                if 'valid_loss' in record:
                    validation.append(record)
    best = min(validation, key=lambda row: row['valid_loss']) if validation else {}
    final = validation[-1] if validation else {}
    return best, final


rows = []
baseline_summary_paths = [Path('outputs/rank_sweep_summary.csv'), Path('outputs/runs/rank_sweep_summary.csv')]
for summary_path in baseline_summary_paths:
    if summary_path.exists():
        baseline_df = pd.read_csv(summary_path)
        baseline_row = baseline_df[baseline_df['rank'] == 4].iloc[0].to_dict()
        rows.append({
            'comparison': 'baseline',
            'pattern_name': 'uniform_r4',
            'total_rank_units': 192,
            'trainable_params': baseline_row.get('trainable_params'),
            'best_valid_loss': baseline_row.get('best_valid_loss'),
            'best_valid_ppl': baseline_row.get('best_valid_ppl'),
            'best_valid_nll_loss': baseline_row.get('best_valid_nll_loss'),
            'best_valid_nll_ppl': baseline_row.get('best_valid_nll_ppl'),
            'best_valid_epoch': baseline_row.get('best_valid_epoch'),
            'final_valid_loss': baseline_row.get('final_valid_loss'),
            'final_valid_nll_loss': baseline_row.get('final_valid_nll_loss'),
            'bleu': baseline_row.get('bleu'),
            'rouge_l': baseline_row.get('rouge_l'),
            'line_bleu': baseline_row.get('line_bleu'),
            'line_rouge_l': baseline_row.get('line_rouge_l'),
            'run_dir': baseline_row.get('run_dir'),
        })
        break

for spec, config_path, run_dir in pattern_config_paths:
    metrics_path = run_dir / 'generations_test.metrics.json'
    params_path = run_dir / 'parameter_report.json'
    metrics = json.loads(metrics_path.read_text()) if metrics_path.exists() else {}
    params = json.loads(params_path.read_text()) if params_path.exists() else {}
    best, final = read_validation_summary(run_dir)
    rank_units = sum(layer['query'] + layer['value'] for layer in spec['rank_pattern'])
    rows.append({
        'comparison': spec['comparison'],
        'pattern_name': spec['pattern_name'],
        'total_rank_units': rank_units,
        'trainable_params': params.get('trainable'),
        'best_valid_loss': best.get('valid_loss'),
        'best_valid_ppl': best.get('valid_ppl'),
        'best_valid_nll_loss': best.get('valid_nll_loss'),
        'best_valid_nll_ppl': best.get('valid_nll_ppl'),
        'best_valid_epoch': best.get('epoch'),
        'final_valid_loss': final.get('valid_loss'),
        'final_valid_nll_loss': final.get('valid_nll_loss'),
        'bleu': metrics.get('bleu'),
        'rouge_l': metrics.get('rouge_l'),
        'line_bleu': metrics.get('line_bleu'),
        'line_rouge_l': metrics.get('line_rouge_l'),
        'run_dir': str(run_dir),
    })

df = pd.DataFrame(rows)
summary_path = Path('outputs/runs/rank_pattern_summary.csv')
summary_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(summary_path, index=False)
display(df)
print('Saved:', summary_path)

In [ ]:
plot_df = df[df['comparison'] != 'baseline'].copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, comparison in zip(axes, ['depth_allocation', 'projection_allocation']):
    subset = plot_df[plot_df['comparison'] == comparison]
    ax.bar(subset['pattern_name'], subset['bleu'])
    if not df[df['pattern_name'] == 'uniform_r4'].empty:
        baseline_bleu = float(df.loc[df['pattern_name'] == 'uniform_r4', 'bleu'].iloc[0])
        ax.axhline(baseline_bleu, linestyle='--', label='uniform_r4')
    ax.set_title(comparison.replace('_', ' ').title())
    ax.set_ylabel('BLEU')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis='y', alpha=0.25)
    ax.legend()
fig.tight_layout()
plot_path = Path('outputs/runs/rank_pattern_summary.png')
fig.savefig(plot_path, dpi=180)
print('Saved:', plot_path)

## 8. Back Up Rank-Pattern Runs To Drive

Copy each completed pattern run, generated configs, summary artifacts, and this notebook into Google Drive so Colab runtime resets do not lose results.

In [ ]:
import shutil

DRIVE_PATTERN_DIR.mkdir(parents=True, exist_ok=True)

for spec, config_path, run_dir in pattern_config_paths:
    if run_dir.exists():
        destination = DRIVE_PATTERN_DIR / run_dir.name
        shutil.copytree(run_dir, destination, dirs_exist_ok=True)
        print('Backed up:', run_dir, '->', destination)
    if config_path.exists():
        config_dest = DRIVE_PATTERN_DIR / 'configs' / config_path.name
        config_dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(config_path, config_dest)
        print('Backed up:', config_path, '->', config_dest)

for artifact_path in [
    Path('outputs/runs/rank_pattern_summary.csv'),
    Path('outputs/runs/rank_pattern_summary.png'),
    Path('colab_rank_pattern_lora.ipynb'),
]:
    if artifact_path.exists():
        shutil.copy2(artifact_path, DRIVE_PATTERN_DIR / artifact_path.name)
        print('Backed up:', artifact_path)

!find /content/drive/MyDrive/lora_rank_patterns -maxdepth 3 -type f | sort | tail -80